# 01 — Data Loading & Validation

Load the EDP Open Data SCADA signals and failure logbook, verify column mapping, and save cleaned parquet files to `data/processed/`.

## Before running

1. Download the **EDP Open Data** dataset from https://edp.com/en/innovation/data  
   (Look for the wind energy / SCADA open data package — zip contains per-turbine CSVs + a logbook CSV.)
2. Place the per-turbine SCADA CSVs in `data/raw/scada/`
3. Place the logbook CSV in `data/raw/logbook/`

Expected structure:
```
data/raw/
  scada/
    T01.csv
    T02.csv
    ...
  logbook/
    logbook.csv
```

In [ ]:
import sys
from pathlib import Path

# add src/ to path so we can import project modules
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import pandas as pd
from src.load import load_all, load_scada, load_logbook

RAW_DIR = repo_root / 'data' / 'raw'
PROCESSED_DIR = repo_root / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load data

In [ ]:
data = load_all(RAW_DIR)
scada   = data['scada']
logbook = data['logbook']

## 2. SCADA inspection

In [ ]:
if scada is not None:
    print('Shape:', scada.shape)
    print('Turbines:', sorted(scada['turbine_id'].unique()) if 'turbine_id' in scada.columns else 'N/A')
    display(scada.head())
    display(scada.dtypes.rename('dtype'))
    display(scada.describe())

In [ ]:
# Date range
if scada is not None and 'timestamp' in scada.columns:
    print('Date range:', scada['timestamp'].min(), '→', scada['timestamp'].max())
    print('Duration:', scada['timestamp'].max() - scada['timestamp'].min())

In [ ]:
# Missing values per column
if scada is not None:
    missing = scada.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if len(missing):
        print('Missing values:')
        display(missing)
    else:
        print('No missing values.')

## 3. Logbook inspection

In [ ]:
if logbook is not None:
    print('Shape:', logbook.shape)
    display(logbook.head(10))
    display(logbook.dtypes.rename('dtype'))

In [ ]:
# Component distribution
if logbook is not None and 'component' in logbook.columns:
    display(logbook['component'].value_counts())

In [ ]:
# Downtime summary
if logbook is not None and 'duration_hours' in logbook.columns:
    print('Total downtime (h):', logbook['duration_hours'].sum())
    print('Events with recorded duration:', logbook['duration_hours'].notna().sum())

## 4. Save processed files

In [ ]:
if scada is not None:
    out = PROCESSED_DIR / 'scada.parquet'
    scada.to_parquet(out, index=False)
    print(f'Saved {out} ({out.stat().st_size/1e6:.1f} MB)')

if logbook is not None:
    out = PROCESSED_DIR / 'logbook.parquet'
    logbook.to_parquet(out, index=False)
    print(f'Saved {out} ({out.stat().st_size/1e6:.1f} MB)')

## 5. Quick sanity plot — power curve

In [ ]:
import matplotlib.pyplot as plt

if scada is not None and {'wind_speed_ms', 'active_power_kw'}.issubset(scada.columns):
    sample = scada.sample(min(10_000, len(scada)), random_state=42)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(sample['wind_speed_ms'], sample['active_power_kw'],
               alpha=0.2, s=5, color='steelblue')
    ax.set_xlabel('Wind Speed (m/s)')
    ax.set_ylabel('Active Power (kW)')
    ax.set_title('Power Curve (sample 10k points)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Failure count by component
if logbook is not None and 'component' in logbook.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    logbook['component'].value_counts().plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Failure Count by Component')
    ax.set_xlabel('Component')
    ax.set_ylabel('Count')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()